##Understand the Requirements: What kind of MCQs? What difficulty levels? Any specific topics or input methods (e.g., text, document upload)?
##Choose a Technology Stack: For a Colab environment, Python is ideal. We might use libraries like gradio or streamlit for a simple UI, or integrate with a large language model (LLM) for question generation.
##Question Generation Logic:
##Manual Input: Allow users to type in questions and options.
##Automated Generation (with LLM): If we decide to use an LLM (like Gemini), we'll need to integrate with its API to generate questions from provided text.
##User Interface (UI): Build a simple interface for:
Inputting content or topics for questions.
Displaying generated MCQs.
Allowing users to edit, save, or export MCQs.
##Error Handling and Edge Cases: Implement robust error handling.

In [ ]:
!pip install --force-reinstall langchain langchain-groq langchain-pinecone langchain-community langchain-huggingface

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.8 MB/s eta 0:00:00
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached langchain_openai-1.2.2-py3-none-any.whl.metadata (3.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.8 MB/s eta 0:00:00
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)


In [2]:
from pinecone import Pinecone
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_4344/1394046864.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [3]:
# @title
# Import the UserData module to access Colab secrets
from google.colab import userdata
from langchain_groq import ChatGroq

# Retrieve the API key from Colab secrets
GROQ_API_KEY = userdata.get('Groq_Api_key')

# Initialize the OpenAI client
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name='mixtral-8x7b-32768')

print("AI client initialized successfully!")

AI client initialized successfully!


### Load a PDF document

First, we'll create a directory named `Docs/` where you can upload your PDF file. Then, we'll use `PyPDFDirectoryLoader` to load the PDF document(s) from this directory.

In [4]:
import os

# Create a directory to store the PDF document
if not os.path.exists('Docs'):
    os.makedirs('Docs')

print("Please upload your PDF file into the 'Docs/' folder in the file browser on the left.")

Please upload your PDF file into the 'Docs/' folder in the file browser on the left.


In [5]:
!pip install pypdf
# Load the PDF document from the 'Docs' directory
loader = PyPDFDirectoryLoader('Docs/')
documents = loader.load()

print(f"Loaded {len(documents)} document(s).")
if documents:
    print("First 500 characters of the first document:")
    print(documents[0].page_content[:500])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 14.7 MB/s eta 0:00:00
Loaded 0 document(s).


### Split Documents into Chunks

To effectively use the document content for question generation, we need to split it into smaller, overlapping chunks. This helps the language model process information in manageable pieces.


In [6]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split the documents
texts = text_splitter.split_documents(documents)

print(f"Split into {len(texts)} chunks.")
if texts:
    print("First chunk content:")
    print(texts[0].page_content[:500])

Split into 0 chunks.


### Create Embeddings

Next, we'll create numerical representations (embeddings) of our text chunks using a HuggingFace embedding model. These embeddings will be stored in a vector store, allowing us to efficiently search and retrieve relevant text segments for question generation.

In [7]:
# Initialize the HuggingFace embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("HuggingFace embeddings model initialized.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace embeddings model initialized.


In [8]:
from google.colab import userdata
import os
from pinecone import Pinecone, ServerlessSpec, NotFoundException, PineconeApiException # Added NotFoundException, PineconeApiException

# Retrieve Pinecone API key and environment from Colab secrets
PINE_API_KEY = userdata.get('PINECONE_API_KEY')
PINE_ENV = userdata.get('PINECONE_ENVIRONMENT') # e.g., 'gcp-starter'

# Set environment variables for Pinecone
os.environ['PINECONE_API_KEY'] = PINE_API_KEY
os.environ['PINECONE_ENVIRONMENT'] = PINE_ENV

# Initialize Pinecone
pinecone = Pinecone(
    api_key=PINE_API_KEY,
    environment=PINE_ENV
)

INDEX_NAME = "test" # Using 'test' as suggested by previous error. Adjust if you have another preferred index.

# Define the expected dimension for the embeddings (from 'sentence-transformers/all-MiniLM-L6-v2')
EXPECTED_DIMENSION = 384

try:
    # Attempt to describe the index. If it fails, NotFoundException is raised.
    index_description = pinecone.describe_index(INDEX_NAME)

    # Index exists, check dimension
    if index_description.dimension != EXPECTED_DIMENSION:
        print(f"Index '{INDEX_NAME}' exists but has incorrect dimension ({index_description.dimension}). Deleting and recreating with dimension {EXPECTED_DIMENSION}.")
        pinecone.delete_index(INDEX_NAME)
        # Recreate after deletion
        pinecone.create_index(
            INDEX_NAME,
            dimension=EXPECTED_DIMENSION,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region=PINE_ENV)
        )
    else:
        print(f"Index '{INDEX_NAME}' already exists with correct dimension ({EXPECTED_DIMENSION}).")

except NotFoundException:
    # Index does not exist, attempt to create it
    print(f"Index '{INDEX_NAME}' not found. Attempting to create a new index with dimension {EXPECTED_DIMENSION}.")
    try:
        pinecone.create_index(
            INDEX_NAME,
            dimension=EXPECTED_DIMENSION, # Dimension of all-MiniLM-L6-v2 embeddings
            metric='cosine', # Common metric for embeddings
            spec=ServerlessSpec(cloud='aws', region=PINE_ENV) # Added spec argument
        )
        print(f"Successfully created index '{INDEX_NAME}'.")
    except PineconeApiException as e:
        # Specifically catch the ALREADY_EXISTS error during creation
        if e.status == 409 and "ALREADY_EXISTS" in str(e.body):
            print(f"Index '{INDEX_NAME}' was created during the process or already exists. Proceeding.")
            # Verify the dimension after assuming it exists
            try:
                index_description = pinecone.describe_index(INDEX_NAME)
                if index_description.dimension != EXPECTED_DIMENSION:
                    print(f"Warning: Index '{INDEX_NAME}' was created but has incorrect dimension ({index_description.dimension}). Please manually delete and recreate it with dimension {EXPECTED_DIMENSION}.")
            except NotFoundException:
                print(f"Warning: Index '{INDEX_NAME}' still not found after conflict. It might be in a transitional state.")
        else:
            raise e # Re-raise any other PineconeAPI exceptions

# Create a Pinecone vector store from the documents
docsearch = PineconeVectorStore.from_documents(texts, embeddings, index_name=INDEX_NAME)

print(f"Pinecone vector store '{INDEX_NAME}' created and loaded with {len(texts)} embeddings.")

Index 'test' already exists with correct dimension (384).
Pinecone vector store 'test' created and loaded with 0 embeddings.


In [9]:
#This function will help us in fetching the top relevent documents from our vector store - Pinecone
def get_similiar_docs(query, k=2):
    similar_docs = index.similarity_search(query, k=k)
    return similar_docs

In [12]:
from langchain.chains.question_answering import load_qa_chain

from langchain_huggingface import HuggingFaceEndpoint

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# @title
# import gradio as gr

# # Define a function to generate MCQs
# def generate_mcqs_from_topic(topic: str):
#     if not topic:
#         return "Please enter a topic to generate MCQs."

#     # Retrieve relevant documents from the vector store
#     # Assuming 'docsearch' is your PineconeVectorStore instance
#     # and 'llm' is your ChatGroq instance from previous cells
#     relevant_docs = docsearch.similarity_search(topic, k=4) # Retrieve top 4 relevant docs

#     # Combine the content of relevant documents
#     context = "\n\n".join([doc.page_content for doc in relevant_docs])

#     if not context:
#         return "No relevant information found for the given topic. Please try a different topic or upload more data."

#     # Construct the prompt for the LLM to generate MCQs
#     prompt = f"""You are an expert in creating multiple-choice questions (MCQs) from provided text.
#     Generate 3 distinct multiple-choice questions with 4 options each (A, B, C, D) and clearly indicate the correct answer.
#     Each question should be based on the following context. Ensure the questions are clear and the options are plausible but only one is correct.

#     Context:
#     {context}

#     Questions:
#     1. Question text?
#     A. Option A
#     B. Option B
#     C. Option C
#     D. Option D
#     Correct Answer: B

#     2. Question text?
#     A. Option A
#     B. Option B
#     C. Option C
#     D. Option D
#     Correct Answer: C

#     3. Question text?
#     A. Option A
#     B. Option B
#     C. Option C
#     D. Option D
#     Correct Answer: A
#     """

#     try:
#         # Use the LLM to generate the MCQs
#         response = llm.invoke(prompt)
#         return response.content
#     except Exception as e:
#         return f"An error occurred while generating MCQs: {e}"

# # Create a Gradio interface
# iface = gr.Interface(
#     fn=generate_mcqs_from_topic,
#     inputs=gr.Textbox(lines=2, placeholder="Enter a topic to generate MCQs (e.g., 'Gated Recurrent Unit')"),
#     outputs=gr.Markdown(), # Display output as Markdown for better formatting
#     title="AI-Powered MCQ Generator",
#     description="Enter a topic, and the AI will generate multiple-choice questions based on the uploaded PDF data.",
#     allow_flagging="never"
# )

# # Launch the interface
# iface.launch(debug=True, share=True)